In [1]:
import mlflow
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

/home/itxkakar/Personal /Coding Playground/Hands-on-Machine-Learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [3]:
mlflow.set_experiment("Deep Learning Experiment")

# IMPORTANT: Enable system metrics monitoring
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

In [4]:
iris = pd.read_csv('Iris.csv')
iris.head()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [5]:
X_raw = iris.iloc[:, 0:4].values
y_raw = iris.iloc[:, -1].values

In [6]:
label_mapping = {label: idx for idx, label in enumerate(np.unique(y_raw))}
y_encoded = np.array([label_mapping[label] for label in y_raw])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
class IrisDataset(Dataset):
    def __init__(self, features, labels):
        # Features must be float32 for PyTorch linear layers
        self.X = torch.tensor(features, dtype=torch.float32)
        # Labels must be long (int64) for nn.CrossEntropyLoss
        self.y = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Instantiate datasets
train_dataset = IrisDataset(X_train_scaled, y_train)
test_dataset = IrisDataset(X_test_scaled, y_test)

# Create DataLoaders for batching during training
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Quick verification check
for batch_X, batch_y in train_loader:
    print("Features shape (Batch, Features):", batch_X.shape) # e.g., torch.Size([16, 4])
    print("Labels shape:", batch_y.shape)                     # e.g., torch.Size([16])
    break

Features shape (Batch, Features): torch.Size([16, 4])
Labels shape: torch.Size([16])


In [10]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.xor_mlp_pytorch = nn.Sequential(
            nn.Linear(4, 768),
            nn.ReLU(),
            nn.Linear(768, 768),
            nn.ReLU(),
            nn.Linear(768, 3),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.xor_mlp_pytorch(x)
        return logits


model = NeuralNetwork().to(device)

In [11]:
# Training parameters
params = {
    "epochs": 5,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "optimizer": "SGD",
    "model_type": "MLP",
    "hidden_units": [768, 768],
}

# Define optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"])

In [12]:
with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params(params)
    input_example = torch.tensor(
        X_train_scaled[:1],
        dtype=torch.float32,
        device=device,
    )

    _original_log_model = mlflow.pytorch.log_model

    def log_model_with_example(*args, **kwargs):
        kwargs.setdefault("input_example", input_example)
        return _original_log_model(*args, **kwargs)

    mlflow.pytorch.log_model = log_model_with_example
    
    for epoch in range(params["epochs"]):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics (every 100 batches)
            if batch_idx % 100 == 0:
                batch_loss = train_loss / (batch_idx + 1)
                batch_acc = 100.0 * correct / total
                mlflow.log_metrics(
                    {"batch_loss": batch_loss, "batch_accuracy": batch_acc},
                    step=epoch * len(train_loader) + batch_idx,
                )

        # Calculate epoch metrics
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = loss_fn(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate and log epoch validation metrics
        val_loss = val_loss / len(test_loader)
        val_acc = 100.0 * val_correct / val_total

        # Log epoch metrics
        mlflow.log_metrics(
            {
                "train_loss": epoch_loss,
                "train_accuracy": epoch_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            },
            step=epoch,
        )
        # Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f"checkpoint_{epoch}")

        print(
            f"Epoch {epoch + 1}/{params['epochs']}, "
            f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

    # Log the final trained model
    model_info = mlflow.pytorch.log_model(model, name="final_model")

2026/11/06 09:22:14 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/11/06 09:22:14 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/11/06 09:22:14 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/11/06 09:22:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


MlflowException: Unsupported signature type for the selected serialization format. If the `serialization_format` argument is set to 'pt2', the input signature must be specified using `TensorSpec`. Please update the model signature or set `serialization_format` to 'pickle'.

In [ ]:
# !mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000